<a href="https://colab.research.google.com/github/carlospucv/recommender_system/blob/rs_v4/RS_UsandoImplicit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo usando KNN (K-Nearest Neighbors)- BASADA EN ID LISTA
Filtrado Colaborativo y Matriz Usuario×Película

**Técnicas**
* Filtrado colaborativo: se basa exclusivamente en el comportamiento de usuarios (listas de películas), sin usar metadatos (género, año, etc.).

* Item-to-item: para cada película “target”, buscamos otras películas cuyos “perfiles de usuario” sean similares.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix

# 1) Prepara tus datos (igual que antes)
movies_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movies.csv')
data      = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/resultados.csv')
data['ID_Lista']    = data['ID_Lista'].astype(str)
data['ID_Pelicula'] = data['ID_Pelicula'].astype(str)

# Filtra listas muy cortas
valid = data.groupby('ID_Lista').filter(lambda grp: len(grp) >= 2)

# Crea códigos numéricos
valid['user_idx']  = valid['ID_Lista'].astype('category').cat.codes
valid['movie_idx'] = valid['ID_Pelicula'].astype('category').cat.codes

# Mapas para volver a IDs
movie_idx_to_id = {
    idx: mid
    for mid, idx in zip(
        valid['ID_Pelicula'],
        valid['movie_idx']
    )
}

# 2) Matriz Usuario×Película en CSR
n_users  = valid['user_idx'].nunique()
n_movies = valid['movie_idx'].nunique()
mat = coo_matrix(
    (np.ones(len(valid)), (valid['user_idx'], valid['movie_idx'])),
    shape=(n_users, n_movies)
).tocsr()

# 3) Entrena el KNN sobre Película×Usuario
item_user = mat.T.tocsr()
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(item_user)

# 4) Función de recomendación
from collections import Counter

def recommend_for_list(id_lista, N=5):
    # Obtén las películas de esa lista
    movies_in_list = valid.loc[valid['ID_Lista'] == id_lista, 'movie_idx'].unique()
    candidates = []
    for midx in movies_in_list:
        # busca N+1 vecinos (el primero es él mismo)
        dists, neigh = knn.kneighbors(item_user[midx], n_neighbors=N+1)
        candidates.extend(neigh[0][1:])  # ignoro la primera posición

    # cuento y excluyo los already seen
    cnt = Counter(candidates)
    for midx in movies_in_list:
        cnt.pop(midx, None)

    top_idxs = [midx for midx, _ in cnt.most_common(N)]
    top_ids  = [movie_idx_to_id[midx] for midx in top_idxs]

    return movies_df[movies_df['id'].astype(str).isin(top_ids)][['id','title']]

# 5) Prueba inmediata
id_lista_prueba = valid['ID_Lista'].iloc[0]
print("Lista de prueba:", id_lista_prueba)
print(recommend_for_list(id_lista_prueba, N=5))


# Modelo usando KNN (K-Nearest Neighbors)- BASADA EN ID PELÍCULA
Filtrado Colaborativo y Matriz Usuario×Película

**Técnicas**
* Filtrado colaborativo: se basa exclusivamente en el comportamiento de usuarios (listas de películas), sin usar metadatos (género, año, etc.).

* Item-to-item: para cada película “target”, buscamos otras películas cuyos “perfiles de usuario” sean similares.

In [ ]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix

# 1) Prepara tus datos (igual que antes)
movies_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movies.csv')
data      = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/resultados.csv')
data['ID_Lista']    = data['ID_Lista'].astype(str)
data['ID_Pelicula'] = data['ID_Pelicula'].astype(str)

# Filtra listas muy cortas
valid = data.groupby('ID_Lista').filter(lambda grp: len(grp) >= 2)

# Crea códigos numéricos
valid['user_idx']  = valid['ID_Lista'].astype('category').cat.codes
valid['movie_idx'] = valid['ID_Pelicula'].astype('category').cat.codes

# Mapas para volver a IDs
movie_idx_to_id = {
    idx: mid
    for mid, idx in zip(
        valid['ID_Pelicula'],
        valid['movie_idx']
    )
}

# 2) Matriz Usuario×Película en CSR
n_users  = valid['user_idx'].nunique()
n_movies = valid['movie_idx'].nunique()
mat = coo_matrix(
    (np.ones(len(valid)), (valid['user_idx'], valid['movie_idx'])),
    shape=(n_users, n_movies)
).tocsr()

# 3) Entrena el KNN sobre Película×Usuario
item_user = mat.T.tocsr()
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(item_user)

# 4) Función de recomendación
from collections import Counter

def recommend_for_list(id_lista, N=5):
    # Obtén las películas de esa lista
    movies_in_list = valid.loc[valid['ID_Lista'] == id_lista, 'movie_idx'].unique()
    candidates = []
    for midx in movies_in_list:
        # busca N+1 vecinos (el primero es él mismo)
        dists, neigh = knn.kneighbors(item_user[midx], n_neighbors=N+1)
        candidates.extend(neigh[0][1:])  # ignoro la primera posición

    # cuento y excluyo los already seen
    cnt = Counter(candidates)
    for midx in movies_in_list:
        cnt.pop(midx, None)

    top_idxs = [midx for midx, _ in cnt.most_common(N)]
    top_ids  = [movie_idx_to_id[midx] for midx in top_idxs]

    return movies_df[movies_df['id'].astype(str).isin(top_ids)][['id','title']]

def recommend_for_movie(movie_id, N=5):
    """
    Dada una película (movie_id, como aparece en tu CSV `movies.csv`),
    devuelve las N películas más similares según cosine sobre Película×Usuario.
    """
    # 1) Mapea el ID original al índice de fila en item_user
    #    valid['movie_idx'] viene de tu DataFrame filtrado
    matches = valid.loc[valid['ID_Pelicula'] == movie_id, 'movie_idx'].unique()
    if len(matches) == 0:
        raise ValueError(f"Película {movie_id!r} no encontrada en los datos filtrados.")
    midx = matches[0]

    # 2) Obtén los vecinos (N+1 porque el primero es la película misma)
    distances, neighbors = knn.kneighbors(item_user[midx], n_neighbors=N+1)

    # 3) Ignora el primero (es ella misma) y toma los siguientes N
    neigh_idxs = neighbors[0][1:]

    # 4) Mapea de índice de fila a ID original y extrae título
    rec_ids = [movie_idx_to_id[n] for n in neigh_idxs]
    return movies_df[movies_df['id'].astype(str).isin(rec_ids)][['id','title']]

# ——— Ejemplo de uso ———
# Supongamos que quieres recomendaciones para la película con id '12345'
recs = recommend_for_movie('19995', N=5)
print(recs)


In [ ]:
# 1) Asegúrate de que los tipos coincidan
movies_df['id']          = movies_df['id'].astype(str)
data['ID_Lista']         = data['ID_Lista'].astype(str)
data['ID_Pelicula']      = data['ID_Pelicula'].astype(str)

# 2) Merge para traer el título junto a IDs
tabla_peliculas_listas = data.merge(
    movies_df[['id', 'title']],    # sólo id y title
    left_on='ID_Pelicula',         # en data
    right_on='id'                  # en movies_df
)[
    ['title', 'id', 'ID_Lista']    # columnas de interés
]

# 3) Renombra para mayor claridad
tabla_peliculas_listas.columns = [
    'Título Película', 'ID Película', 'ID Lista'
]

# 4) Muestra la tabla
display(tabla_peliculas_listas)
# —o si prefieres—
# print(tabla_peliculas_listas.head(50))


# Modelo entrenamiento Keras

**Técnicas empleadas:**

  1. **Formulación como clasificación multilabel:**
    - Cada lista de usuario es tratada como un vector binario 𝑦 ∈{ 0,1 } ^𝑀
    donde 𝑀 es el número total de películas.
    - El input   𝑥 ∈ { 0, 1 }^𝑀   marca un subconjunto de 2–5 películas de la misma lista.

2. **Red neuronal densa:**

  - Dos capas ocultas (128 y 64 neuronas) con ReLU para aprender interacciones no lineales entre películas.

  - Capa de salida sigmoid multilabel para predecir la probabilidad de pertenencia de cada película a la lista.

3. **Función de pérdida y métricas:**

  - Binary Crossentropy: adecuada para clasificación multilabel.

  - Precision & Recall: más informativas que la exactitud global en problemas desbalanceados.

4. **Regularización y prevención de overfitting**

  - **EarlyStopping** con paciencia para detener el entrenamiento cuando la validación deja de mejorar.

5. División train/test 50/50

  - Garantiza que el modelo se pruebe en listas nunca vistas.

**Fundamento teórico:**

- El filtrado colaborativo clásico se basa en similitud de usuarios o ítems; aquí lo reformulamos como problema de clasificación multilabel, lo que permite usar la potencia de las redes neuronales para capturar patrones complejos de co-aparición de películas en listas de usuarios.

- Las capas densas aprenden representaciones latentes de combinaciones de películas, mientras que la capa sigmoide permite emitir recomendaciones como probabilidades independientes.

- El uso de precision y recall como métricas refleja mejor la capacidad real del modelo para recomendar películas relevantes (recall) sin inundar con falsos positivos (precision).



In [ ]:
# ----------------------------
# 1. Imports
# ----------------------------
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, GlobalAveragePooling1D,
    Dense
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

# ----------------------------
# 2. Cargar datos
# ----------------------------


movies_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movies.csv') # columnas: id, title, …
data      = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/resultados.csv')  # columnas: ID_Lista, ID_Pelicula

# Aseguramos tipos string para los IDs
data['ID_Lista']    = data['ID_Lista'].astype(str)
data['ID_Pelicula'] = data['ID_Pelicula'].astype(str)
movies_df['id']     = movies_df['id'].astype(str)

# ----------------------------
# 3. Filtrar listas pequeñas
# ----------------------------
min_size = 2
valid_lists = data.groupby('ID_Lista') \
                  .filter(lambda grp: len(grp) >= min_size)

# ----------------------------
# 4. Preparar mapeos
# ----------------------------
# Conjunto de listas y de películas
all_list_ids   = valid_lists['ID_Lista'].unique().tolist()
all_movie_ids  = valid_lists['ID_Pelicula'].unique().tolist()

# Diccionarios ID ↔ índice
movie_id_to_index = {mid: i for i, mid in enumerate(sorted(all_movie_ids))}
index_to_movie_id = {i: mid for mid, i in movie_id_to_index.items()}

list_to_movies = valid_lists.groupby('ID_Lista')['ID_Pelicula'] \
                            .apply(list) \
                            .to_dict()

num_movies = len(movie_id_to_index)

# ----------------------------
# 5. División train / test
# ----------------------------
train_list_ids, test_list_ids = train_test_split(
    all_list_ids,
    test_size=0.5,
    random_state=42
)

# ----------------------------
# 6. Función para crear X, Y
# ----------------------------
def create_dataset(list_ids):
    X, Y = [], []
    for lid in list_ids:
        movies = list_to_movies[lid]
        if len(movies) < 2:
            continue

        # elegimos entre 2 y 5 elementos para X
        k = min(random.randint(2,5), len(movies))
        sample_in = random.sample(movies, k)

        # vector binario de entrada
        x = [movie_id_to_index[mid] for mid in sample_in]
        # vector binario de salida (todas las películas de la lista)
        y = np.zeros(num_movies, dtype='float32')
        for mid in movies:
            y[movie_id_to_index[mid]] = 1.0

        X.append(x)
        Y.append(y)
    return X, np.vstack(Y)

# Crear datasets
X_train, Y_train = create_dataset(train_list_ids)
X_test,  Y_test  = create_dataset(test_list_ids)

# ----------------------------
# 7. Pad sequences
# ----------------------------
maxlen = max(len(seq) for seq in X_train + X_test)
X_train_padded = pad_sequences(X_train, maxlen=maxlen, padding='post')
X_test_padded  = pad_sequences(X_test,  maxlen=maxlen, padding='post')

# ----------------------------
# 8. Definir pérdida focal (opcional)
# ----------------------------
def binary_focal_loss(gamma=2., alpha=.25):
    def loss_fn(y_true, y_pred):
        eps   = K.epsilon()
        y_pred = K.clip(y_pred, eps, 1-eps)
        pt     = tf.where(K.equal(y_true,1), y_pred, 1-y_pred)
        return -K.mean(alpha * K.pow(1-pt, gamma) * K.log(pt))
    return loss_fn

# ----------------------------
# 9. Métricas personalizadas
# ----------------------------
def custom_precision(y_true, y_pred):
    thresh = 0.4
    y_pred_bin = tf.cast(y_pred > thresh, tf.float32)
    tp = K.sum(y_true * y_pred_bin)
    pp = K.sum(y_pred_bin)
    return tp / (pp + K.epsilon())

def custom_recall(y_true, y_pred):
    thresh = 0.4
    y_pred_bin = tf.cast(y_pred > thresh, tf.float32)
    tp = K.sum(y_true * y_pred_bin)
    pos = K.sum(y_true)
    return tp / (pos + K.epsilon())

# ----------------------------
# 10. Construir el modelo
# ----------------------------
embedding_size = 32

input_layer = Input(shape=(maxlen,), dtype='int32')
embed      = Embedding(input_dim=num_movies, output_dim=embedding_size)(input_layer)
profile    = GlobalAveragePooling1D()(embed)
dense1     = Dense(128, activation='relu')(profile)
dense2     = Dense(64,  activation='relu')(dense1)
output     = Dense(num_movies, activation='sigmoid')(dense2)

model = Model(inputs=input_layer, outputs=output)
model.compile(
    optimizer='adam',
    loss=binary_focal_loss(gamma=2, alpha=.25),
    metrics=[custom_precision, custom_recall]
)

# ----------------------------
# 11. Entrenar
# ----------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_padded, Y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.1,
    callbacks=[early_stop]
)

# ----------------------------
# 12. Evaluar
# ----------------------------
results = model.evaluate(X_test_padded, Y_test, batch_size=16)
print(f"Pérdida de test:       {results[0]:.4f}")
print(f"Precisión de test:     {results[1]:.4f}")
print(f"Recall de test:        {results[2]:.4f}")

# ----------------------------
# 13. Función de recomendación
# ----------------------------
def get_recommendations(input_movie_ids, top_n=5):
    idxs = [movie_id_to_index[mid] for mid in input_movie_ids]
    x = pad_sequences([idxs], maxlen=maxlen, padding='post')
    preds = model.predict(x)[0]
    # ordenar por probabilidad descendente y excluir inputs
    ranked = np.argsort(preds)[::-1]
    recs = [i for i in ranked if i not in idxs][:top_n]
    rec_ids = [index_to_movie_id[i] for i in recs]
    return movies_df[movies_df['id'].isin(rec_ids)][['id','title']]

# Ejemplo de uso:
sample_list = train_list_ids[0]
entrada = random.sample(list_to_movies[sample_list], 3)
print("Input IDs:", entrada)
print(get_recommendations(entrada, top_n=5))
